# **Урок 8**. Распознавание лиц с InsightFace

В лекции мы с вами узнали, что процесс распознавания лиц в видеопотоке состоит из нескольких шагов: 
- детекции и трекинга лиц, 
- выбора лучших кадров,
- распознавания атрибутов, в том числе Liveness,
- выравнивания/нормализации лица (например, с помощью гомографии по точкам),
- извлечения дескриптора,
- проверки похожести на фотографии из базы/на эталон.

[`InsightFace`](https://github.com/deepinsight/insightface) — это библиотека, которая даём нам готовые решения для лиц почти для всех этих шагов, а также предоставляет возможность самим экспериментировать с обучением моделей. 

Давайте в этой практике познакомимся с библиотекой и её возможностями.  

## **План**

1. Тестируем FaceAnalysis
2. Собираем своё приложение

## **1. Тестируем FaceAnalysis**

Ставим библиотеку через pip:

In [ ]:
%pip install -U insightface

In [ ]:
import cv2
import sys
import numpy as np
import matplotlib.pyplot as plt

import insightface
from insightface.app import FaceAnalysis
from insightface.data import get_image as ins_get_image

Для демо моделек нам понадобится [python package InsightFace](https://github.com/deepinsight/insightface/tree/master/python-package). 

В нём есть `FaceAnalysis` — полноценное приложение, которое может полностью обработать кадр в плане распознавания лиц. Давайте потестируем его.

In [ ]:
# Создаём app для распознавания лиц, name тут определяет всё: и детектор, и тип выравнивания, и атрибуты, и сеть извлечения дескриптора
# В providers выбираем на чём будет запускаться onnxruntime
app = FaceAnalysis(name='buffalo_s', providers=['CPUExecutionProvider'])

После создания `app`-а видим, какие модельки в нём собрались и какие они ожидают размеры входа. 

У детектора он может варьироваться, так что он дополнительно устанавливается в методе `prepare`.

In [ ]:
# Готовим изображение
# https://github.com/deepinsight/insightface/blob/master/python-package/insightface/app/face_analysis.py#L47
# ctx_id — номер девайса, на котором будет инференс происходить
app.prepare(ctx_id=0, det_size=(320, 320))

In [ ]:
img = cv2.imread('./assets/celebs.jpg')

fig = plt.figure(figsize=(10, 15))
plt.axis('off')
plt.imshow(img[:,:,::-1])

In [ ]:
# Берём картинку в обработку с помощью нашего app
# Обратите внимание, что app.get ожидает BGR 
faces = app.get(img)

In [ ]:
# И смотрим на результат
print('Num of faces: {}'.format(len(faces)))

In [ ]:
print(faces[0].keys())
print(faces[0]['bbox'])
print(len(faces[0]['embedding']))

In [ ]:
# https://github.com/deepinsight/insightface/blob/master/python-package/insightface/app/face_analysis.py#L79
# Взяла фукцию отдельно, так как при вызове из библиотеки берётся её старая версия с deprication
def draw_on(img, faces, verbose=True):
    dimg = img.copy()
    for i in range(len(faces)):
        face = faces[i]
        box = face.bbox.astype(int)
        color = (0, 0, 255)
        cv2.rectangle(dimg, (box[0], box[1]), (box[2], box[3]), color, 2)
        if face.kps is not None:
            kps = face.kps.astype(int)
            for l in range(kps.shape[0]):
                color = (0, 0, 255)
                if l == 0 or l == 3:
                    color = (0, 255, 0)
                cv2.circle(dimg, (kps[l][0], kps[l][1]), 1, color, 2)
                
        if (face.gender is not None) and (face.age is not None) and (verbose):
            cv2.putText(dimg,'%s,%d'%(face.sex,face.age), (box[0]-1, box[1]-4),cv2.FONT_HERSHEY_COMPLEX,0.7,(0,255,0),1)

    return dimg

In [ ]:
# Отрисуем результаты
res_img = draw_on(img, faces)

fig = plt.figure(figsize=(10, 15))
plt.axis('off')
plt.imshow(res_img[:, :, ::-1])

Что мы видим:
- почти все лица нашлись;
- ключевые точки лиц выглядят отлично;
- к определению возраста и гендера, по-моему, есть большие вопросы :)

Если мы сделаем всё то же самое, но размер входа увеличим, то получится лучше:

In [ ]:
app.prepare(ctx_id=0, det_size=(640, 640))
faces_2 = app.get(img)
res_img_2 = draw_on(img, faces_2)

fig = plt.figure(figsize=(10, 15))
plt.axis('off')
plt.imshow(res_img_2[:, :, ::-1])

А теперь давайте попробуем что-нибудь посложнее:

In [ ]:
img = cv2.imread('./assets/largest_selfie.jpg')
app.prepare(ctx_id=0, det_size=(640, 640))
faces = app.get(img)
res_img = draw_on(img, faces, verbose=False)

fig = plt.figure(figsize=(15, 25))
plt.axis('off')
plt.imshow(res_img[:, :, ::-1])

Выглядит не очень впечатляюще, давайте возьмём лучшее, что есть в `InsightFace`:

In [ ]:
app = FaceAnalysis(name='buffalo_l', providers=['CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))
faces = app.get(img)
res_img_l = draw_on(img, faces, verbose=False)

fig = plt.figure(figsize=(15, 25))
plt.axis('off')
plt.imshow(res_img_l[:, :, ::-1])

Выглядит уже намного лучше, а что, если ещё и порог снизить?

In [ ]:
app = FaceAnalysis(name='buffalo_l', providers=['CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640), det_thresh=0.25)
faces = app.get(img)
res_img_l = draw_on(img, faces, verbose=False)

fig = plt.figure(figsize=(15, 25))
plt.axis('off')
plt.imshow(res_img_l[:, :, ::-1])

Итак, мы поняли, что
- в `InsightFace` есть набор готовых моделек для работы с лицами разной жирности;
- эти модельки подготовлены в `onnx`-формате;
- работают эти наборы как единое целое, ничего заменить вы в них не можете, только аргументы детектора перебирать;
- с помощью `FaceAnalysis` мы можем делать полную обработку кадра: находить все лица, их ключевые точки, определять атрибуты и вычислять дескрипторы;
- конечно, чем жирнее будут модельки, тем лучше вы получите результаты.

`InsightFace pypackage` также позволяет загружать модельки по отдельности, давайте так тоже научимся делать.  

## **2. Собираем своё приложение**

На самом деле распознавание в самом простом случае можно построить всего на двух моделях:
- детектор лиц (возможно, с ключевыми точками)
- модель для извлечения дескриптора лица. 

Начнём с них.

Скачаем веса для набора моделей `antelopev2` из [Model Zoo](https://github.com/deepinsight/insightface/tree/master/python-package#model-zoo).

In [ ]:
import onnxruntime as ort 

In [ ]:
# Прописываем пути к моделькам
det_model_path = './antelopev2/scrfd_10g_bnkps.onnx'
rec_model_path = './antelopev2/glintr100.onnx'

In [ ]:
# Создаём детектор
# https://github.com/deepinsight/insightface/blob/master/python-package/insightface/model_zoo/scrfd.py#L72
det_sess = ort.InferenceSession(str(det_model_path))
detector = insightface.model_zoo.scrfd.SCRFD(det_model_path, session=det_sess)

# Устанавливаем его параметры для инференса
detector.prepare(ctx_id=0, nms_thresh=0.4, det_thresh=0.2, input_size=(640, 640))

In [ ]:
# Создаём модель для распознавания
# https://github.com/deepinsight/insightface/blob/master/python-package/insightface/model_zoo/arcface_onnx.py#L19
rec_sess = ort.InferenceSession(str(rec_model_path))
embedder = insightface.model_zoo.arcface_onnx.ArcFaceONNX(rec_model_path, session=rec_sess)

Посмотрим, что нам отдаёт модель детектора:

In [ ]:
img = cv2.imread('./assets/celebs.jpg')
det_res = detector.detect(img, max_num=30)

In [ ]:
print(type(det_res))
print(det_res[0].shape)
print(det_res[1].shape)

In [ ]:
print(det_res[0][0])
print(det_res[1][0])

Мы получаем кортеж из списка `bbox`-ов лиц с их скорами уверенности и списка 5 ключевых точек всех задетекченных лиц. 

Давайте их отрисуем:

In [ ]:
def draw_bboxes_points(img, det_result):
    dimg = img.copy()
    for i in range(len(det_result[0])):
        box = det_result[0][i][:4].astype(int)
        score = det_result[0][i][4]
        color = (0, 0, 255)
        cv2.rectangle(dimg, (box[0], box[1]), (box[2], box[3]), color, 2)
        cv2.putText(dimg, f'Score {score:.3f}', (box[0]-1, box[1]-4), cv2.FONT_HERSHEY_COMPLEX, 0.7, (0,255,0), 1)
        
        kps = det_result[1][i].astype(int)
        for l in range(kps.shape[0]):
            cv2.circle(dimg, (kps[l][0], kps[l][1]), 1, color, 2)

    return dimg

In [ ]:
vis_img = draw_bboxes_points(img, det_res)

fig = plt.figure(figsize=(15, 25))
plt.axis('off')
plt.imshow(vis_img[:, :, ::-1])

Выглядит супер, на этот раз мы нашли вообще все лица и понимаем, какой порог на скор детектора нам подойдёт. 

Далее давайте посмотрим, как работать с извлечением дескриптора лица. 

Сперва лицо нам надо подготовить: 
- вырезать его из полного кадра, 
- провести выравнивание (`alignment`), 
- сделать ресайз к целевому размеру - `(112, 112)`.

Вся подготовка уже реализована в `InsightFace`, нам только надо упаковать bbox и ключевые точки в нужный `dict`:

In [ ]:
# https://github.com/deepinsight/insightface/blob/master/python-package/insightface/app/common.py#L5
# Берём из нашего списка первое, самое уверенное лицо
# На самом деле тут даже бокс передавать не обязательно, точек достаточно
face = insightface.app.common.Face(bbox=det_res[0][0], kps=det_res[1][0])

In [ ]:
# Давайте посмотрим, что это за лицо
face_crop = img[int(det_res[0][0][1]):int(det_res[0][0][3]), int(det_res[0][0][0]):int(det_res[0][0][2])]
plt.imshow(face_crop[:, :, ::-1])

Отлично, считать дескриптор будем для Джулии Робертс :)

Далее картинку и информацию о позиционировании лица на ней передаём в готовый метод сети `.get()`. 

В нём по ключевым точкам будет проводиться [выравнивание](https://github.com/deepinsight/insightface/blob/master/python-package/insightface/utils/face_align.py#L27). Давайте посмотрим, как эти функции утроены и какой результат дают:

In [ ]:
# https://github.com/deepinsight/insightface/blob/master/python-package/insightface/utils/face_align.py
from skimage import transform as trans
arcface_dst = np.array(
    [[38.2946, 51.6963], [73.5318, 51.5014], [56.0252, 71.7366],
     [41.5493, 92.3655], [70.7299, 92.2041]],
    dtype=np.float32)


def estimate_norm(lmk, image_size=112, mode='arcface'):
    assert lmk.shape == (5, 2)
    assert image_size % 112==0 or image_size % 128==0
    
    if image_size % 112==0:
        ratio = float(image_size)/112.0
        diff_x = 0
    else:
        ratio = float(image_size)/128.0
        diff_x = 8.0 * ratio
    
    dst = arcface_dst * ratio
    dst[:,0] += diff_x
    tform = trans.SimilarityTransform()
    tform.estimate(lmk, dst)
    M = tform.params[0:2, :]
    return M

# Основная функция, которая предобрабатывает изображение
def norm_crop(img, landmark, image_size=112, mode='arcface'):
    M = estimate_norm(landmark, image_size, mode) # Считаем матрицу преобразования
    warped = cv2.warpAffine(img, M, (image_size, image_size), borderValue=0.0) # Выполняем преобразование к целевому размеру
    return warped

# Получим выровненный кроп
aimg = norm_crop(img, landmark=face.kps)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 15))
ax[0].axis('off')
ax[0].imshow(face_crop[:, :, ::-1])
ax[1].axis('off')
ax[1].imshow(aimg[:, :, ::-1])

Вот такое выровненный квадратик и пойдет дальше на извлечение дескриптора. 

In [ ]:
# https://github.com/deepinsight/insightface/blob/master/python-package/insightface/model_zoo/arcface_onnx.py#L65
# А извлекаться он будет тут
embedding = embedder.get(img, face)

In [ ]:
print(embedding.shape)

Получили вектор длины 512, который описывает лицо Джулии.

Посчитаем аналогичные для всех остальных:

In [ ]:
embeddings = []

# Пробегаем по всем детекциям и их точкам, выравниваем лица, считаем их дескрипторы
for i in range(len(det_res[0])):
    face = insightface.app.common.Face(bbox=det_res[0][i], kps=det_res[1][i])
    embedding = embedder.get(img, face)
    embeddings.append(embedding)
    
embeddings = np.array(embeddings)

In [ ]:
# Проверяем shape
embeddings.shape

А теперь давайте посмотрим, на кого из них больше всего похожа Джулия в фильме «Красотка».

Аналогично задетектируем на фото лицо и ключевые точки:

In [ ]:
etalon = cv2.imread('./assets/pretty_woman.jpg')
det_etalon = detector.detect(etalon, max_num=30)

vis_etalon = draw_bboxes_points(etalon, det_etalon)
crop_etalon = etalon[int(det_etalon[0][0][1]):int(det_etalon[0][0][3]), int(det_etalon[0][0][0]):int(det_etalon[0][0][2])]

fig, ax = plt.subplots(1, 2, figsize=(10, 15))
ax[0].axis('off')
ax[0].imshow(vis_etalon[:, :, ::-1])
ax[1].axis('off')
ax[1].imshow(crop_etalon[:, :, ::-1])

Посмотрим, как будет выглядеть его выровненная версия:

In [ ]:
face = insightface.app.common.Face(bbox=det_etalon[0][0], kps=det_etalon[1][0])
aetalon = norm_crop(etalon, landmark=face.kps)

plt.imshow(aetalon[:, :, ::-1])

И считаем его дескриптор:

In [ ]:
etalon_embedding = embedder.get(etalon, face)

Теперь посчитаем попарные расстояния между этим эмбеддингом и всеми остальными.

Считать будем косинусное расстояние. Перед подсчётом расстояния дескрипторы надо обязательно нормализовать, чтобы они оказались на единичной сфере.

Чем ближе будут наши векторы на единичной сфере, тем меньше между ними угол, тем больше будет cosine, ведь в нуле он принимает максимальное значение.

In [ ]:
def cosine_similarity(descr_left, descr_right):
    left_normalized = descr_left / np.linalg.norm(descr_left)
    right_normalized = descr_right / np.linalg.norm(descr_right)
    return left_normalized @ right_normalized

In [ ]:
# Заводим список для всех значений 
metrics = []

# Пробегаем по списку и считаем все значения
for emb in embeddings:
    cos_dist = cosine_similarity(etalon_embedding, emb)
    metrics.append(cos_dist)
    
metrics = np.array(metrics)

Посмотрим в каком же элементе у нас будет наибольшее значение. Мы ожидаем, что в 0-ом, ведь именно там живет дескриптор Джулии на церемонии. 

In [ ]:
print(np.argmax(metrics))
print(metrics)

Это победа, Джулия, действительно, больше всего похожа на саму себя :)

А мы с вами в этом разделе:
- поняли, как звать по отдельности готовые модельки, которые есть в `InsightFace`;
- увидели, как по ключевым точкам происходит выравнивание лица;
- смогли извлечь дескрипторы для всех найденных на фото лиц;
- убедились, что наиболее близкими друг к другу по косинусному расстоянию оказываются дескрипторы, которые принадлежат одному и тому же человеку.

Супер, давайте подведём финальные итоги.

## **Итоги**

В этой практике мы:
- познакомились с репозиторием `InsightFace`;
- потестировали готовые пайплайны для end-to-end обработки кадров с лицами;
- собрали собственный небольшой пайплайн для обработки кадров и с его помощью извлекли дескрипторы лиц;
- научились считать расстояния между дескрипторами и убедились, что чем более похожи лица, тем ближе будут их дескрипторы.

Вы ещё поиграетесь с распознаванием лиц в домашнем задании, а мы на сегодня закончим. 

See you!